# Transformers & self-attention

_Modern Deep Learning & AI_

**Every token looks at every other token and pulls in what matters. No loops, all at once.**

A Transformer reads a whole sentence at once. It does not crawl word by word.

     The trick is self-attention. Every word builds three small vectors: a query (what am I looking for?), a key (what do I offer?), and a value (what I carry).

---

This notebook is a practice scaffold for the **Transformers & self-attention** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F

def attention(Q, K, V):
    # Q,K,V: (batch, seq_len, d). One query/key/value vector per token.
    d = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d ** 0.5   # (batch, seq, seq)
    weights = F.softmax(scores, dim=-1)           # each row sums to 1
    return weights @ V, weights                   # blended values

torch.manual_seed(0)
batch, seq, d = 1, 4, 8                            # 4 tokens, dim 8
x = torch.randn(batch, seq, d)
# learnable projections build Q, K, V from the same input (self-attention)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
Q, K, V = x @ Wq, x @ Wk, x @ Wv
out, attn = attention(Q, K, V)
print("output shape:", out.shape)                 # (1, 4, 8)
print("row 0 weights sum:", attn[0, 0].sum().item())  # ~1.0

## Visualize the data & results

_In the sentence The animal didnt cross the street because it was too tired, what does it attend to?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# scaled dot-product attention over a REAL sentence
tokens = ["The","animal","didnt","cross","the","street",
          "because","it","was","too","tired"]
seq, d = len(tokens), 16
rng = np.random.default_rng(7)
X = np.stack([np.random.default_rng(1000+i).standard_normal(d)
              for i in range(seq)])    # one embedding per token
X[7] = 0.30*X[7] + 0.95*X[1]           # coreference: "it" embedding ~ "animal"
X[4] = 0.30*X[4] + 0.95*X[0]           # "the" ~ "The"

Wq = np.eye(d) + 0.15*rng.standard_normal((d, d))   # near-identity projections
Wk = np.eye(d) + 0.15*rng.standard_normal((d, d))
Q, K = X @ Wq, X @ Wk
scores = Q @ K.T / np.sqrt(d)
scores -= scores.max(axis=1, keepdims=True)
W = np.exp(scores); W /= W.sum(axis=1, keepdims=True)   # each row sums to 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(W, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(seq)); ax.set_xticklabels(tokens, rotation=90)
ax.set_yticks(range(seq)); ax.set_yticklabels(tokens)
ax.set_title("Self-attention: row it peaks on animal")
fig.colorbar(im, ax=ax)
plt.show()